In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#read datasets with pandas

#Training Data
train_df = pd.read_csv('train.csv')

#Testing Data
test_df = pd.read_csv('test.csv')

In [ ]:
#describe the training dataframe
train_df.describe()

In [ ]:
#describe the test dataframe
test_df.describe()

In [ ]:
#show the train dataframe head
train_df.head()

In [ ]:
#show the test dataframe head
test_df.head()

In [ ]:
#drop passengerId column
train_df = train_df.drop(columns=['PassengerId'])
test_df = test_df.drop(columns=['PassengerId'])

In [ ]:
train_df = train_df.drop_duplicates()

In [ ]:
import re

#Define title and family name extraction function using regex
def extract_titanic_names(full_name):
    if not isinstance(full_name, str):
        return pd.Series([None, None])
    
    # Regex breakdown for "Last, Title. First":
    # ^(?P<family_name>[^,]+) -> Everything before the comma is the family name
    # ,\s*(?P<title>[^\.]+)\. -> The title is between the comma+space and the dot
    pattern = r'^(?P<family_name>[^,]+),\s*(?P<title>[^\.]+)\.'
    
    match = re.search(pattern, full_name.strip())
    
    if match:
        family_name = match.group('family_name').strip()
        title = match.group('title').strip()
        return pd.Series([family_name, title])
    
    return pd.Series([None, None])

# 3. Apply the function to create new columns
train_df[['Family_Name', 'Title']] = train_df['Name'].apply(extract_titanic_names)
test_df[['Family_Name', 'Title']] = test_df['Name'].apply(extract_titanic_names)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
#drop Name column after extracting title and family_name
train_df = train_df.drop(columns='Name')
test_df = test_df.drop(columns='Name')

In [ ]:
#define a function that lists different titles with their frequencies
def list_titles_with_counts(df):
    return df['Title'].value_counts()

In [ ]:
list_titles_with_counts(train_df)

In [ ]:
list_titles_with_counts(test_df)

In [ ]:
#define a function that replaces non-frequent titles with 'rare' to reduce on-hot encoding sparsity
def clean_titles(df):
    # Map rare or french variations to standard equivalents
    title_mapping = {
        'Mlle': 'Miss',
        'Ms': 'Miss',
        'Mme': 'Mrs',
    }
    df['Title'] = df['Title'].replace(title_mapping)
    
    # Group anything else very rare into a single 'Rare' bucket
    rare_titles = ['Dr', 'Rev', 'Major', 'Col', 'Don', 'Lady', 'Sir', 'Capt', 'the Countess', 'Jonkheer']
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    
    return df

#replace non-frequent title values with 'Rare'
train_df = clean_titles(train_df)
test_df = clean_titles(test_df)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
#define a function that lists the number of missing values for each feature to help clean the data
def features_missing_values(df):
    return df.isnull().sum()

In [ ]:
# Check for missing values in all columns of training data
features_missing_values(train_df)

In [ ]:
# Check for missing values in all columns of test data
features_missing_values(test_df)

In [ ]:
#since sibsp and parch have no missing values , create a family_size feature that's equal to the sum of all related people to a single individual
train_df['Family_Size'] = train_df['Parch'] + train_df['SibSp'] + 1
test_df['Family_Size'] = test_df['Parch'] + test_df['SibSp'] + 1
columns_to_drop = ['Parch' ,'SibSp']
train_df = train_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
#Bin the family size feature into 3 categories: Alone , Small , Large
#Training set
conditions = [
    train_df['Family_Size'] == 1,
    train_df['Family_Size'].between(2, 4)
]
choices = ['Alone', 'Small']

train_df['Family_Category'] = np.select(conditions, choices, default='Large')

train_df

In [ ]:
#Bin the family size feature into 3 categories: Alone , Small , Large
#Test Set
conditions = [
    test_df['Family_Size'] == 1,
    test_df['Family_Size'].between(2, 4)
]
choices = ['Alone', 'Small']

test_df['Family_Category'] = np.select(conditions, choices, default='Large')

test_df

In [ ]:
#Extract Letters from cabin numbers:
# 1. Fill missing cabins with 'U' for Unknown
train_df['Cabin'] = train_df['Cabin'].fillna('U')
test_df['Cabin'] = test_df['Cabin'].fillna('U')

# 2. Extract the first character as the Deck letter
train_df['Cabin_Deck'] = train_df['Cabin'].str.slice(0, 1)
test_df['Cabin_Deck'] = test_df['Cabin'].str.slice(0,1)

#Check new decks and their counts in train set
print(train_df['Cabin_Deck'].value_counts())

In [ ]:
#Check new decks and their counts in test set
print(test_df['Cabin_Deck'].value_counts())

In [ ]:
#drop cabin feature
train_df = train_df.drop(columns=['Cabin'])
test_df = test_df.drop(columns=['Cabin'])

In [ ]:
# Count how many rows share the same ticket number across your dataset
train_df['Ticket_Frequency'] = train_df['Ticket'].map(train_df['Ticket'].value_counts())
test_df['Ticket_Frequency'] =   test_df['Ticket'].map(test_df['Ticket'].value_counts())

In [ ]:
#drop ticket feature
train_df = train_df.drop(columns=['Ticket'])
test_df = test_df.drop(columns=['Ticket'])

In [ ]:
#Create a numeric to categorical age converter to use in the master pipeline
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class TitleAgeImputerAndBinner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.title_medians_ = {}
        self.global_median_ = 0.0

    def fit(self, X, y=None):
        # Make a copy to avoid mutating the original dataframe
        X_ = X.copy()
        
        # Calculate the median age for each title strictly from the training data
        self.title_medians_ = X_.groupby('Title')['Age'].median().to_dict()
        
        # Calculate a global fallback median
        self.global_median_ = X_['Age'].median()
        return self

    def transform(self, X):
        X_ = X.copy()
        
        # 1. Fill missing ages using the learned training medians
        for title, median_val in self.title_medians_.items():
            mask = (X_['Title'] == title) & (X_['Age'].isnull())
            X_.loc[mask, 'Age'] = median_val
            
        # 2. Fill any remaining NaNs with the global median
        X_['Age'] = X_['Age'].fillna(self.global_median_)
        
        # 3. Bin into Age Categories
        bins = [-1, 12, 18, 60, np.inf]
        labels = ['Child', 'Teenager', 'Adult', 'Senior']
        X_['Age_Category'] = pd.cut(X_['Age'], bins=bins, labels=labels)
        
        # 4. Drop the continuous numeric age column
        X_ = X_.drop(columns=['Age'])
        
        return X_

In [ ]:
#Create a family survival rate encoder that calculates family survival rate on each Fold of the K-Folds
from sklearn.base import BaseEstimator, TransformerMixin

class FamilySurvivalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.family_survival_map = {}

    def fit(self, X, y=None):
        # Ensure we have a DataFrame and y is provided
        if y is None:
            raise ValueError("FamilySurvivalEncoder requires y during fit.")
        
        temp_df = X.copy()
        temp_df['Survived'] = y
        # Calculate mean survival per family name from training data
        self.family_survival_map = temp_df.groupby('Family_Name')['Survived'].mean().to_dict()
        return self

    def transform(self, X):
        X_out = X.copy()
        # Map the training survival rates to the current set
        X_out['Family_Survival_Rate'] = X_out['Family_Name'].map(self.family_survival_map)
        # Fill unseen/new families with a neutral baseline (0.5)
        X_out['Family_Survival_Rate'] = X_out['Family_Survival_Rate'].fillna(0.5)
        return X_out

In [ ]:
#Create sub-pipelines to scale continuous-valued features and one-hot encode class-based features
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler , OneHotEncoder

num_features = ['Fare', 'Family_Survival_Rate' , 'Ticket_Frequency']
cat_features = ['Pclass', 'Age_Category' , 'Sex' , 'Embarked' , 'Title' , 'Family_Category' , 'Cabin_Deck']

#numeric transformer sub pipeline to fill missing numeric values with the mean of train dataset and then scale them
numeric_transformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='median')), #calculate mean and fill missing values
    ('scaler' , StandardScaler()) #scale features
])

#categorical transformer that fills missing values with the mode of train dataset and then one hot encode them
categorical_transformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='most_frequent')), #calculate mode and fill missing values
    ('onehot' , OneHotEncoder(handle_unknown='ignore' , sparse_output=False)) #sparse_output=False means return an array with the 0s
])

#the 2 piplines combined into a preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

In [ ]:
#Create the final logistic regression pipeline
from sklearn.linear_model import LogisticRegression

master_pipeline = Pipeline(steps=[
    ('age_processor' , TitleAgeImputerAndBinner()),
    ('family_survival', FamilySurvivalEncoder()),
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42 , C=0.1))
])

In [ ]:
#evaluate our baseline model using cross_val_score on 5 folds
from sklearn.model_selection import cross_val_score

X_train = train_df.drop(columns=['Survived'])
y_train = train_df['Survived']

cross_val_scores = cross_val_score(master_pipeline , X_train , y_train , cv=5 , scoring='accuracy') 

print("Cross-Validation Accuracy Scores:", cross_val_scores)
print("Logitic Regression Mean Accuracy:", cross_val_scores.mean())

In [ ]:
#Test RandomForest and XGBoost models against this dataset
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# 1. Test Random Forest
master_pipeline.set_params(model=RandomForestClassifier(random_state=42 , max_depth=2 , n_estimators=100))
rf_scores = cross_val_score(master_pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"Random Forest Mean Accuracy: {rf_scores.mean():.4f}")

In [ ]:
# 2. Test XGBoost
master_pipeline.set_params(model=XGBClassifier(
    random_state=42,
    eval_metric='logloss' ,
    learning_rate=0.05 ,
    max_depth=4 ,
    n_estimators=150,
    reg_alpha=0.1,         # L1 regularization (similar to Lasso)
    reg_lambda=1.0         # L2 regularization (similar to Ridge)
 )
)
xgb_scores = cross_val_score(master_pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"XGBoost Mean Accuracy: {xgb_scores.mean():.4f}")